# RL upper-bound improver for unknotting numbers

This notebook is designed to live **inside a GitHub-style repository** and run
**without Google Drive**.

Expected local layout:

```text
repo-root/
├─ upper_bound_unknotting_v6_local.ipynb
├─ requirements.txt
├─ README.md
├─ data/
│  └─ unknotting.xlsx
├─ models/
│  └─ best_model.zip                 # optional
├─ training_data/                    # optional
│  ├─ hard_unknots.csv
│  ├─ very_hard_unknots.csv
│  └─ random_diagrams.csv
└─ outputs/
```

The notebook does the following:

1. loads `data/unknotting.xlsx`
2. fills missing Jones vectors from the PD presentation when needed
3. finds knots whose unknotting number is given by a range such as `[2,3]`
4. inflates the diagram
5. flips one crossing at a time
6. runs the RL unknotter / reducer
7. computes the Jones vector of the reduced knot
8. identifies all matching knots in the database, allowing mirrors
9. takes the **maximum** matching upper bound
10. improves the original upper bound when possible by setting it to `[old_lower, 1 + matched_upper_bound]`
11. saves the workbook after each processed knot

If no pretrained model is present, the notebook will try to train one from
local training files. If those are absent too, it falls back to PD data already
present in `unknotting.xlsx`, so the repository can still run in a self-contained
way.

The Jones-vector convention used here is:

`[lowest power of q, highest power of q, a, b, c, ...]`

with all intermediate coefficients included, including zeros.


In [1]:
# Local setup (run once per environment, if needed)
# %pip install -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 628.9/628.9 kB 10.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.2/406.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.5/94.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.9/146.9 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.3/63.3 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.0/165.0 kB 14.0 MB/s eta 

In [2]:

import os, re, json, ast, math, random, csv, glob, time
from pathlib import Path
from dataclasses import dataclass
from fractions import Fraction
from collections import defaultdict
from typing import Iterable, Optional, List, Tuple

import numpy as np
import pandas as pd

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

from tqdm.auto import tqdm

import snappy
from spherogram import Link

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
# Local repository paths (no Google Drive needed)
from pathlib import Path

CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = None
for cand in CANDIDATE_ROOTS:
    if (cand / "data").exists() or (cand / "models").exists() or (cand / "training_data").exists():
        REPO_ROOT = cand
        break
if REPO_ROOT is None:
    REPO_ROOT = Path.cwd()

DATA_DIR = REPO_ROOT / "data"
MODELS_DIR = REPO_ROOT / "models"
TRAINING_DIR = REPO_ROOT / "training_data"
OUT_DIR = REPO_ROOT / "outputs"

for folder in [DATA_DIR, MODELS_DIR, TRAINING_DIR, OUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

XLSX_PATH = DATA_DIR / "unknotting.xlsx"
if not XLSX_PATH.exists():
    raise FileNotFoundError(
        f"Missing workbook: {XLSX_PATH}\n"
        "Place your database at repo-root/data/unknotting.xlsx"
    )

BASE = REPO_ROOT

print("REPO_ROOT:", REPO_ROOT)
print("Workbook :", XLSX_PATH)
print("MODELS   :", MODELS_DIR)
print("TRAINING :", TRAINING_DIR)
print("OUTPUTS  :", OUT_DIR)


Mounted at /content/drive
BASE: /content/drive/MyDrive/crossing-reduction-unknotting
Workbook: /content/drive/MyDrive/crossing-reduction-unknotting/unknotting.xlsx
OUT_DIR: /content/drive/MyDrive/crossing-reduction-unknotting/upper_bound_improver_outputs


In [4]:
# ----------------------------
# Configuration
# ----------------------------
# Which unresolved knots to process:
#   PROCESS_MODE = "all"              -> all range-valued unknotting entries, i.e. all [a,b] with a != b
#   PROCESS_MODE = "first_n"          -> first N range-valued entries
#   PROCESS_MODE = "slice"            -> entries [START_INDEX:END_INDEX] among range-valued entries
#   PROCESS_MODE = "bounds_eq"        -> only entries with unknotting number exactly [TARGET_LOWER, TARGET_UPPER]
#   PROCESS_MODE = "bounds_eq_slice"  -> first filter to [TARGET_LOWER, TARGET_UPPER], then take [START_INDEX:END_INDEX]
#   PROCESS_MODE = "bounds_neq"       -> explicitly all [a,b] with a != b (same target set as "all")
#   PROCESS_MODE = "bounds_neq_slice" -> explicitly all [a,b] with a != b, then take [START_INDEX:END_INDEX]
PROCESS_MODE = "slice"

TARGET_LOWER = 1
TARGET_UPPER = 2

START_INDEX = 0          # 0-based within the filtered target list
END_INDEX = 10            # exclusive; e.g. 100 means "first 100"
FIRST_N = 100              # used only when PROCESS_MODE == "first_n"

NUM_VARIANTS_PER_KNOT = 12
BACKTRACK_STEPS_MIN = 6
BACKTRACK_STEPS_MAX = 8
RIII_STEPS_MAX = 20

UNKNOTTER_EPISODES_PER_FLIP = 1
UNKNOTTER_MAX_STEPS = 500

TRAIN_IF_MODEL_MISSING = True
TRAIN_STEPS_IF_NEEDED = 20000

# Saving / output
SAVE_AFTER_EACH_TARGET = True
SAVE_EVERY_KNOT = True          # robust alias used in the main loop
WRITE_UPDATED_COPY = False
MAKE_TIMESTAMPED_BACKUP = False
# No backup file and no secondary updated copy: only overwrite unknotting.xlsx

# Only trust Jones identification when the reduced PD lies in the database range.
MIN_DATABASE_CROSSINGS = 1
MAX_DATABASE_CROSSINGS = 13

# Optional safety valve for very long runs:
MAX_FLIPS_PER_VARIANT = None   # set to an integer to cap crossing flips per inflated variant

MODEL_PATH_CANDIDATES = [
    BASE / "best_model.zip",
    BASE / "ppo_knot_rl_spherogram_continued.zip",
    OUT_DIR / "best_model.zip",
]

# Training data sources from the original notebook / paper pipeline
GCS_CSV_PATH_MAIN = "gs://gdm-unknotting/hard_unknots.csv"
GCS_CSV_PATH_VERY = "gs://gdm-unknotting/very_hard_unknots.csv"

# Optional local extras: only used if present
LOCAL_EXTRA_FILES = [
    BASE / "random_diagrams.csv",
    BASE / "random_diagrams.txt",
    BASE / "hard_unknots.csv",
    BASE / "very_hard_unknots.csv",
]


In [5]:

# ----------------------------
# Helpers: workbook / parsing
# ----------------------------
_int_pat = re.compile(r'-?\d+')

def pick_first_existing(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def parse_pd_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, list):
        return [[int(y) for y in q] for q in x]
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return None
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, list) and all(isinstance(q, (list, tuple)) and len(q) == 4 for q in obj):
            return [[int(y) for y in q] for q in obj]
    except Exception:
        pass
    items = re.findall(r'[Xx]\s*\[([^\]]+)\]', s)
    if items:
        out = []
        for it in items:
            nums = [int(z.strip()) for z in it.split(',')]
            if len(nums) != 4:
                return None
            out.append(nums)
        return out
    return None

def parse_vector_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, (list, tuple)):
        try:
            return [int(v) for v in x]
        except Exception:
            return None
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return None
    if s.startswith("[") and s.endswith("]"):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, (list, tuple)):
                return [int(z) for z in v]
        except Exception:
            pass
    nums = _int_pat.findall(s)
    if not nums:
        return None
    return [int(z) for z in nums]

def ensure_minmax_coeffs(v):
    if v is None:
        return None
    v = [int(x) for x in v]
    if len(v) < 3:
        return None
    mn, mx = v[0], v[1]
    coeffs = v[2:]
    if len(coeffs) != abs(mx - mn) + 1:
        return None
    return mn, mx, coeffs

def strip_leading_trailing_zeros(coeffs):
    coeffs = list(map(int, coeffs))
    i, j = 0, len(coeffs)
    while i < j and coeffs[i] == 0:
        i += 1
    while j > i and coeffs[j-1] == 0:
        j -= 1
    out = coeffs[i:j]
    return out if out else [0]

def canon_coeff_key(coeffs):
    return tuple(strip_leading_trailing_zeros(coeffs))

def canon_coeff_key_mirror(coeffs):
    return tuple(reversed(strip_leading_trailing_zeros(coeffs)))

def span_abs(mn, mx):
    return abs(int(mx) - int(mn))

def parse_unknotting_entry(x):
    """
    Returns dict with:
      kind: 'missing' | 'exact' | 'range' | 'other'
      lower, upper: ints or None
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {"kind": "missing", "lower": None, "upper": None, "raw": x}
    if isinstance(x, (int, np.integer)):
        n = int(x)
        return {"kind": "exact", "lower": n, "upper": n, "raw": x}
    if isinstance(x, float) and float(x).is_integer():
        n = int(x)
        return {"kind": "exact", "lower": n, "upper": n, "raw": x}

    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return {"kind": "missing", "lower": None, "upper": None, "raw": x}

    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, (list, tuple)) and len(obj) == 2:
            a, b = int(obj[0]), int(obj[1])
            if a == b:
                return {"kind": "exact", "lower": a, "upper": b, "raw": x}
            return {"kind": "range", "lower": min(a,b), "upper": max(a,b), "raw": x}
    except Exception:
        pass

    nums = [int(z) for z in _int_pat.findall(s)]
    if len(nums) == 1:
        return {"kind": "exact", "lower": nums[0], "upper": nums[0], "raw": x}
    if len(nums) >= 2:
        a, b = nums[0], nums[1]
        if a == b:
            return {"kind": "exact", "lower": a, "upper": b, "raw": x}
        return {"kind": "range", "lower": min(a,b), "upper": max(a,b), "raw": x}

    return {"kind": "other", "lower": None, "upper": None, "raw": x}

def format_unknotting(lower, upper):
    if lower is None and upper is None:
        return None
    if lower is None or upper is None:
        return None
    return str([int(lower), int(upper)])


In [6]:

# ----------------------------
# Load workbook and identify columns
# ----------------------------
df = pd.read_excel(XLSX_PATH)

knot_col  = pick_first_existing(df, ["knot_id", "name", "knot", "id"])
jones_col = pick_first_existing(df, ["jones_vector", "jones_polynomial_vector"])
pd_col    = pick_first_existing(df, ["pd_presentation", "pd_notation", "pd", "pd_code"])
u_col     = pick_first_existing(df, ["unknotting_number", "unknotting", "u"])

print("Columns:")
print("  knot_col :", knot_col)
print("  jones_col:", jones_col)
print("  pd_col   :", pd_col)
print("  u_col    :", u_col)

if knot_col is None or pd_col is None or u_col is None:
    raise ValueError("Could not identify the required knot / PD / unknotting-number columns.")

if jones_col is None:
    jones_col = "jones_vector"
    df[jones_col] = None
    print(f"Created missing Jones column: {jones_col}")

display(df.head())


Columns:
  knot_col : knot_id
  jones_col: jones_vector
  pd_col   : pd_presentation
  u_col    : unknotting_number


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,knot_id,jones_vector,pd_presentation,unknotting_number
0,0_1,1,NaN,0
1,3_1,"[1, 4, 1, 0, 1, -1]","[[1,5,2,4],[3,1,4,6],[5,3,6,2]]",1
2,4_1,"[-2, 2, 1, -1, 1, -1, 1]","[[4,2,5,1],[8,6,1,5],[6,3,7,4],[2,7,3,8]]",1
3,5_1,"[2, 7, 1, 0, 1, -1, 1, -1]","[[2,8,3,7],[4,10,5,9],[6,2,7,1],[8,4,9,3],[10,...",2
4,5_2,"[1, 6, 1, -1, 2, -1, 1, -1]","[[1,5,2,4],[3,9,4,8],[5,1,6,10],[7,3,8,2],[9,7...",1


In [7]:

# ----------------------------
# Jones polynomial code from the attached notebook
# ----------------------------
def poly_add(p, q):
    r = dict(p)
    for e, c in q.items():
        r[e] = r.get(e, 0) + c
        if r[e] == 0:
            del r[e]
    return r

def poly_mul(p, q):
    r = {}
    for e1, c1 in p.items():
        for e2, c2 in q.items():
            e = e1 + e2
            r[e] = r.get(e, 0) + c1 * c2
    return {e:c for e,c in r.items() if c != 0}

def poly_monom(exp, coeff=1):
    return {int(exp): int(coeff)}

def poly_scale(p, s):
    return {e: c*s for e,c in p.items() if c*s != 0}

class DSU:
    def __init__(self):
        self.p = {}
    def find(self, x):
        if x not in self.p:
            self.p[x] = x
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[rb] = ra
    def n_components(self):
        return len({self.find(x) for x in self.p})

def bracket_from_pd(pd):
    n = len(pd)
    Delta = poly_add(poly_scale(poly_monom(2), -1), poly_scale(poly_monom(-2), -1))

    labels = set()
    for (a,b,c,d) in pd:
        labels.update([a,b,c,d])

    total = {}
    for mask in range(1 << n):
        dsu = DSU()
        for x in labels:
            dsu.find(x)

        a_count = 0
        b_count = 0
        for i, (a,b,c,d) in enumerate(pd):
            if ((mask >> i) & 1) == 0:
                a_count += 1
                dsu.union(a, b)
                dsu.union(c, d)
            else:
                b_count += 1
                dsu.union(b, c)
                dsu.union(d, a)

        loops = dsu.n_components()
        mon = poly_monom(a_count - b_count, 1)

        factor = {0: 1}
        for _ in range(loops - 1):
            factor = poly_mul(factor, Delta)

        total = poly_add(total, poly_mul(mon, factor))
    return total

def crossing_sign_pd(quad):
    a,b,c,d = quad
    return 1 if (a < c) == (b < d) else -1

def jones_string_from_pd(pd_list):
    if not pd_list:
        return None, "Empty PD"
    pdq = [tuple(map(int, q)) for q in pd_list]
    try:
        br = bracket_from_pd(pdq)
        w = sum(crossing_sign_pd(q) for q in pdq)
        sign = -1 if ((-3*w) % 2) else 1
        norm = poly_scale(poly_monom(-3*w, 1), sign)
        normed = poly_mul(norm, br)

        jt = {}
        for eA, c in normed.items():
            eT = Fraction(-eA, 4)
            jt[eT] = jt.get(eT, 0) + c
        jt = {e:c for e,c in jt.items() if c != 0}

        terms = []
        for e in sorted(jt.keys(), reverse=True):
            if e.denominator != 1:
                raise ValueError(f"Non-integral exponent encountered: {e}")
            c = jt[e]
            k = e.numerator
            if k == 0:
                mon = ""
            elif k == 1:
                mon = "t"
            else:
                mon = f"t^{k}"
            if mon == "":
                term = f"{c}"
            else:
                if c == 1:
                    term = mon
                elif c == -1:
                    term = "-" + mon
                else:
                    term = f"{c}*{mon}"
            terms.append(term)

        if not terms:
            return "0", None

        s = terms[0]
        for t in terms[1:]:
            if t.startswith("-"):
                s += " - " + t[1:]
            else:
                s += " + " + t
        return s, None
    except Exception as e:
        return None, repr(e)

def parse_jones_string_to_dict(jstr):
    if jstr is None:
        return None
    s = str(jstr).strip()
    if s == "" or s == "0":
        return {}

    s = s.replace(" - ", " + -")
    parts = [p.strip() for p in s.split(" + ") if p.strip()]
    poly = {}
    for term in parts:
        term = term.replace(" ", "")
        if "*t" in term:
            c_str, mon = term.split("*", 1)
            coeff = int(c_str)
        elif term.startswith("t") or term.startswith("-t"):
            coeff = -1 if term.startswith("-t") else 1
            mon = term[1:] if term.startswith("-t") else term
        else:
            coeff = int(term)
            exp = 0
            poly[exp] = poly.get(exp, 0) + coeff
            continue

        if mon == "t":
            exp = 1
        elif mon.startswith("t^"):
            exp = int(mon[2:])
        else:
            raise ValueError(f"Bad monomial format: {mon}")
        poly[exp] = poly.get(exp, 0) + coeff
    return {e:c for e,c in poly.items() if c != 0}

def poly_dict_to_knotinfo_vector(poly):
    if poly is None:
        return None
    if len(poly) == 0:
        return [0, 0, 0]
    mn, mx = min(poly.keys()), max(poly.keys())
    coeffs = [int(poly.get(e, 0)) for e in range(mn, mx + 1)]
    return [int(mn), int(mx)] + coeffs

def jones_vector_from_pd(pd_list):
    jstr, err = jones_string_from_pd(pd_list)
    if err is not None:
        return None, err
    poly = parse_jones_string_to_dict(jstr)
    vec = poly_dict_to_knotinfo_vector(poly)
    return vec, None


In [8]:

# ----------------------------
# Fill missing Jones vectors in the database itself if needed
# ----------------------------
missing_before = 0
filled = 0
errors = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Filling missing Jones vectors"):
    vec = parse_vector_cell(row.get(jones_col))
    if ensure_minmax_coeffs(vec) is not None:
        continue

    missing_before += 1
    pd_list = parse_pd_cell(row.get(pd_col))
    if pd_list is None:
        errors.append((idx, row.get(knot_col), "missing/bad PD"))
        continue

    if len(pd_list) > 20:
        # the direct bracket expansion is exponential in crossing number
        # so we skip very large PDs here; the improvement pipeline only
        # needs Jones vectors for rows that are actually matched later.
        continue

    new_vec, err = jones_vector_from_pd(pd_list)
    if new_vec is not None:
        df.at[idx, jones_col] = str(new_vec)
        filled += 1
    else:
        errors.append((idx, row.get(knot_col), err))

print("Missing Jones entries before:", missing_before)
print("Filled directly from PD:", filled)
print("Unfilled / errors:", len(errors))
if errors[:10]:
    print("Sample errors:", errors[:10])


Filling missing Jones vectors:   0%|          | 0/12966 [00:00<?, ?it/s]

Missing Jones entries before: 1
Filled directly from PD: 0
Unfilled / errors: 1
Sample errors: [(0, '0_1', 'missing/bad PD')]


In [9]:

# ----------------------------
# Build Jones lookup from the workbook
# The requested rule is:
#   if multiple knots have the same Jones polynomial,
#   take the MAX over all matching upper bounds
# ----------------------------
lookup = defaultdict(list)

for idx, row in df.iterrows():
    vec = parse_vector_cell(row.get(jones_col))
    parsed = ensure_minmax_coeffs(vec)
    if parsed is None:
        continue

    uk = parse_unknotting_entry(row.get(u_col))
    if uk["upper"] is None:
        continue

    mn, mx, coeffs = parsed
    sp = span_abs(mn, mx)
    rec = {
        "row_index": int(idx),
        "knot": row.get(knot_col),
        "upper": int(uk["upper"]),
        "lower": None if uk["lower"] is None else int(uk["lower"]),
        "mirror": False,
    }
    lookup[(sp, canon_coeff_key(coeffs))].append({**rec, "mirror": False})
    lookup[(sp, canon_coeff_key_mirror(coeffs))].append({**rec, "mirror": True})

print("Lookup keys:", len(lookup))


Lookup keys: 16831


In [10]:
# ----------------------------
# Training / RL utilities, adapted from the uploaded notebook
# ----------------------------
import io
import gzip

# Candidate local model files
MODEL_PATH_CANDIDATES = [
    MODELS_DIR / "best_model.zip",
    MODELS_DIR / "ppo_knot_rl_spherogram_continued.zip",
    OUT_DIR / "best_model.zip",
]

# Optional local training files
LOCAL_EXTRA_FILES = [
    TRAINING_DIR / "hard_unknots.csv",
    TRAINING_DIR / "very_hard_unknots.csv",
    TRAINING_DIR / "random_diagrams.csv",
    TRAINING_DIR / "random_diagrams.txt",
    DATA_DIR / "hard_unknots.csv",
    DATA_DIR / "very_hard_unknots.csv",
]

def parse_link_strict(s: str):
    s = s.strip()
    try:
        obj = ast.literal_eval(s)
    except Exception as e:
        raise ValueError(f"Could not parse PD/DT string: {s[:80]}") from e
    if isinstance(obj, list):
        return Link(obj)
    raise ValueError("Parsed training example is not a list-like PD/DT object.")

def clean_pd_lines(lines: Iterable[str], max_keep: int | None = None) -> List[str]:
    good = []
    for s in lines:
        try:
            _ = parse_link_strict(s)
            good.append(s.strip())
            if max_keep and len(good) >= max_keep:
                break
        except Exception:
            continue
    return good

def crossings(link: Link) -> int:
    return len(link.crossings)

def is_trivial_zero(link: Link) -> bool:
    return crossings(link) == 0

def riii_shuffle_only_link(link: Link, k: int, tries_per_move: int = 20):
    from spherogram.links import simplify as _simp
    list_fn  = getattr(_simp, "possible_type_III_moves", None)
    apply_fn = getattr(_simp, "reidemeister_III", None)
    if list_fn is None or apply_fn is None:
        return link, 0

    L = link
    done = 0
    for _ in range(k):
        moves = list_fn(L)
        if not moves:
            break
        tries = min(tries_per_move, len(moves))
        c0 = crossings(L)
        success = False
        for tri in random.sample(moves, tries):
            apply_fn(L, tri)
            if crossings(L) == c0:
                success = True
                break
        if not success:
            break
        done += 1
    return L, done

def read_first_col_local(path: str, has_header: bool = True, encoding: str = "utf-8") -> list[str]:
    out = []
    with open(path, "r", encoding=encoding, newline="") as f:
        rdr = csv.reader(f)
        if has_header:
            next(rdr, None)
        for row in rdr:
            if row:
                out.append(row[0].strip())
    return out

def workbook_pd_lines(max_keep: int | None = None) -> list[str]:
    raw = []
    for _, row in df.iterrows():
        pd_list = parse_pd_cell(row.get(pd_col))
        if pd_list is not None:
            raw.append(json.dumps(pd_list))
            if max_keep is not None and len(raw) >= max_keep:
                break
    return raw

@dataclass
class EnvCfg:
    max_steps: int = UNKNOTTER_MAX_STEPS
    step_penalty: float = 0.05
    reward_finish: float = 10.0
    allow_backtrack: bool = True
    cap_max: int = 8
    w_delta: float = 1.0
    w_uphill: float = 0.5
    w_potential: float = 0.02

class SphKnotEnv(gym.Env):
    def __init__(self, pd_lines: list[str], cfg: EnvCfg):
        super().__init__()
        self.cfg = cfg
        self.pd_lines = pd_lines
        self.rng = random.Random(SEED)
        self.num_actions = 4 if self.cfg.allow_backtrack else 3
        self.action_space = spaces.MultiDiscrete(
            np.array([self.num_actions, self.cfg.cap_max + 1], dtype=np.int64)
        )
        self.obs_dim = 6
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.obs_dim,), dtype=np.float32
        )
        self.L = None
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False
        self._blocked = [False, False, False, False]

    def _reset_blocks(self):
        self._blocked = [False, False, False, False]

    def _map_blocked_mode(self, mode: int) -> int:
        m = mode % self.num_actions
        for _ in range(self.num_actions):
            if not self._blocked[m]:
                return m
            m = (m + 1) % self.num_actions
        return min(3, self.num_actions - 1)

    def _obs(self):
        c = crossings(self.L)
        try:
            comps = len(self.L.link_components)
        except Exception:
            comps = 1
        tmp = Link(self.L.PD_code())
        try:
            reduced = tmp.simplify(mode="basic")
        except TypeError:
            reduced = tmp.simplify()
        can_reduce = 1.0 if (reduced and crossings(tmp) < c) else 0.0
        recently_dropped = float(self._last_drop)
        after_backtrack = 1.0 if self._after_backtrack else 0.0
        return np.array([c, comps, can_reduce, recently_dropped, after_backtrack, self._steps], dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False
        self._reset_blocks()
        pd_str = self.rng.choice(self.pd_lines)
        self.L = parse_link_strict(pd_str)
        return self._obs(), {}

    def step(self, action):
        mode = int(action[0])
        cap = int(action[1]) if len(np.array(action).shape) > 0 else 0
        mode = self._map_blocked_mode(mode)
        self._steps += 1
        old_c = crossings(self.L)
        reward = -self.cfg.step_penalty
        terminated = False
        truncated = False
        info = {"crossings": old_c, "mode": mode, "cap": cap}

        try:
            if mode == 0:
                try:
                    self.L.simplify(mode="basic")
                except TypeError:
                    self.L.simplify()
            elif mode == 1:
                self.L.backtrack(steps=max(1, cap))
                self._after_backtrack = True
            elif mode == 2:
                self.L, _ = riii_shuffle_only_link(self.L, max(1, cap))
            else:
                self.L.simplify(mode="level")
        except Exception:
            self._blocked[mode] = True

        new_c = crossings(self.L)
        delta = old_c - new_c
        self._last_drop = max(0, delta)
        reward += self.cfg.w_delta * delta
        if delta < 0:
            reward += self.cfg.w_uphill * delta

        if is_trivial_zero(self.L):
            reward += self.cfg.reward_finish
            terminated = True

        if self._steps >= self.cfg.max_steps:
            truncated = True

        info["crossings"] = new_c
        return self._obs(), reward, terminated, truncated, info

def load_training_pd_lines():
    raw = []

    for path in LOCAL_EXTRA_FILES:
        if path.exists():
            try:
                extra = read_first_col_local(str(path), has_header=True)
                raw += extra
                print(f"Loaded local extra {path.name}: {len(extra)}")
            except Exception as e:
                print(f"Could not load {path.name}: {e}")

    if not raw:
        raw = workbook_pd_lines()
        print(f"Falling back to workbook PD data: {len(raw)} examples")

    pd_lines = clean_pd_lines(raw, max_keep=None)
    random.Random(SEED).shuffle(pd_lines)
    if not pd_lines:
        raise RuntimeError(
            "No valid PD/DT strings available for training. "
            "Add local CSV/TXT files under training_data/ or ensure unknotting.xlsx contains PD data."
        )
    return pd_lines

def make_single_env(pd_list, cfg: EnvCfg):
    pd_str = json.dumps(pd_list)
    pd_lines_single = [pd_str]
    def _make():
        return SphKnotEnv(pd_lines_single, cfg)
    return DummyVecEnv([_make])

def run_unknotter_on_pd(pd_list, model, cfg: EnvCfg, episodes: int = 3, return_best_pd: bool = False):
    vec_env = make_single_env(pd_list, cfg)
    success = False
    best_crossings_global = len(pd_list)
    best_pd_global = [list(q) for q in pd_list]

    for _ in range(episodes):
        obs = vec_env.reset()
        best_crossings_ep = best_crossings_global
        best_pd_ep = best_pd_global

        for _step in range(cfg.max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, rewards, dones, infos = vec_env.step(action)
            info = infos[0]
            cr = info.get("crossings", None)
            try:
                cur_pd = [list(q) for q in vec_env.envs[0].L.PD_code()]
            except Exception:
                cur_pd = None

            if cr is not None and cr < best_crossings_ep and cur_pd is not None:
                best_crossings_ep = cr
                best_pd_ep = cur_pd

            if cr == 0:
                success = True
                if cur_pd is not None:
                    best_crossings_ep = 0
                    best_pd_ep = cur_pd
                break
            if dones[0]:
                break

        if best_crossings_ep < best_crossings_global:
            best_crossings_global = best_crossings_ep
            best_pd_global = best_pd_ep
        if success:
            break

    vec_env.close()
    if return_best_pd:
        return success, best_crossings_global, best_pd_global
    return success, best_crossings_global


In [11]:
# ----------------------------
# Load or train PPO model
# ----------------------------
cfg = EnvCfg(max_steps=UNKNOTTER_MAX_STEPS, allow_backtrack=True)

# Robust fallback in case the config cell was edited or executed out of order.
if "MODEL_PATH_CANDIDATES" not in globals():
    MODEL_PATH_CANDIDATES = [
        BASE / "best_model.zip",
        BASE / "ppo_knot_rl_spherogram_continued.zip",
        OUT_DIR / "best_model.zip",
    ]

best_model_path = None
for path in MODEL_PATH_CANDIDATES:
    if Path(path).exists():
        best_model_path = Path(path)
        break

if best_model_path is not None:
    model = PPO.load(str(best_model_path), device="auto")
    print("Loaded existing model:", best_model_path)
else:
    if not TRAIN_IF_MODEL_MISSING:
        raise FileNotFoundError(
            "No trained model found in MODEL_PATH_CANDIDATES, and TRAIN_IF_MODEL_MISSING=False."
        )
    print("No existing model found. Training a fresh one.")
    print("Searched these locations:")
    for p in MODEL_PATH_CANDIDATES:
        print("  -", p)

    pd_lines_train = load_training_pd_lines()
    vec_env = DummyVecEnv([lambda: SphKnotEnv(pd_lines_train, cfg)])
    model = PPO(
        "MlpPolicy",
        vec_env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=256,
        n_epochs=10,
        gamma=0.995,
        gae_lambda=0.97,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        seed=SEED,
        verbose=1,
    )
    model.learn(total_timesteps=TRAIN_STEPS_IF_NEEDED, progress_bar=True)
    best_model_path = OUT_DIR / "best_model.zip"
    model.save(str(best_model_path))
    vec_env.close()
    print("Saved trained model to:", best_model_path)

Loaded existing model: /content/drive/MyDrive/crossing-reduction-unknotting/best_model.zip


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:

# ----------------------------
# Inflation and flip helpers
# ----------------------------
def flip_crossing_quad(quad):
    a, b, c, d = quad
    return [b, c, d, a]

def generate_inflated_variants(pd_list,
                               num_variants=10,
                               backtrack_steps_min=6,
                               backtrack_steps_max=8,
                               riii_steps_max=20):
    variants = []
    L0 = Link(pd_list)

    for _ in range(num_variants):
        pd0 = L0.PD_code()
        if isinstance(pd0, list):
            L = Link(pd0)
        else:
            L = Link([[int(getattr(e, "label", e)) for e in vtx] for vtx in pd0])

        steps = random.randint(backtrack_steps_min, backtrack_steps_max)
        try:
            L.backtrack(steps=steps, prob_type_1=0.35, prob_type_2=0.65)
        except Exception:
            pass
        try:
            L, _ = riii_shuffle_only_link(L, min(riii_steps_max, steps))
        except Exception:
            pass

        try:
            pd_new = [list(q) for q in L.PD_code()]
            variants.append(pd_new)
        except Exception:
            variants.append([list(q) for q in pd_list])

    if not variants:
        variants = [[list(q) for q in pd_list]]
    return variants

def generate_single_flip_variants(pd_list):
    out = []
    n = len(pd_list)
    for i in range(n):
        flipped = []
        for j, quad in enumerate(pd_list):
            flipped.append(flip_crossing_quad(quad) if i == j else list(quad))
        out.append((i, flipped))
    return out

def match_jones_vector_to_database(vec):
    parsed = ensure_minmax_coeffs(vec)
    if parsed is None:
        return []
    mn, mx, coeffs = parsed
    sp = span_abs(mn, mx)
    return lookup.get((sp, canon_coeff_key(coeffs)), [])

def best_upper_bound_from_matches(matches):
    if not matches:
        return None
    uppers = [m["upper"] for m in matches if m.get("upper") is not None]
    if not uppers:
        return None
    return max(uppers)


In [13]:
# ----------------------------
# Choose target rows
# ----------------------------
all_range_targets = []
for idx, row in df.iterrows():
    uk = parse_unknotting_entry(row.get(u_col))
    if uk["kind"] == "range":
        all_range_targets.append({
            "row_index": int(idx),
            "knot": row.get(knot_col),
            "lower": int(uk["lower"]),
            "upper": int(uk["upper"]),
        })

print("All range-valued unknotting rows (all [a,b] with a != b):", len(all_range_targets))

bounds_targets = [
    t for t in all_range_targets
    if t["lower"] == TARGET_LOWER and t["upper"] == TARGET_UPPER
]
print(f"Rows with unknotting range [{TARGET_LOWER},{TARGET_UPPER}]:", len(bounds_targets))

neq_targets = list(all_range_targets)
print("Rows with non-exact unknotting range [a,b], a != b:", len(neq_targets))

if PROCESS_MODE == "all":
    targets = list(all_range_targets)
elif PROCESS_MODE == "first_n":
    targets = list(all_range_targets[:FIRST_N])
elif PROCESS_MODE == "slice":
    targets = list(all_range_targets[START_INDEX:END_INDEX])
elif PROCESS_MODE == "bounds_eq":
    targets = list(bounds_targets)
elif PROCESS_MODE == "bounds_eq_slice":
    targets = list(bounds_targets[START_INDEX:END_INDEX])
elif PROCESS_MODE == "bounds_neq":
    targets = list(neq_targets)
elif PROCESS_MODE == "bounds_neq_slice":
    targets = list(neq_targets[START_INDEX:END_INDEX])
else:
    raise ValueError(f"Unknown PROCESS_MODE: {PROCESS_MODE}")

print("Selected targets:", len(targets))
if len(targets) > 0:
    preview = pd.DataFrame(targets[:10])
    display(preview)
else:
    print("No targets selected.")


All range-valued unknotting rows (all [a,b] with a != b): 7601
Rows with unknotting range [1,2]: 5001
Rows with non-exact unknotting range [a,b], a != b: 7601
Selected targets: 900


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,row_index,knot,lower,upper
0,928,12a_127,2,3
1,945,12a_144,3,4
2,946,12a_145,3,4
3,947,12a_146,3,4
4,948,12a_147,2,4
5,949,12a_148,2,4
6,951,12a_150,2,4
7,953,12a_152,2,3
8,954,12a_153,2,3
9,957,12a_156,2,4


In [13]:
# ----------------------------
# Main improvement loop
# ----------------------------
# Robust defaults in case cells were run out of order
if "SAVE_EVERY_KNOT" not in globals():
    SAVE_EVERY_KNOT = True
if "MIN_DATABASE_CROSSINGS" not in globals():
    MIN_DATABASE_CROSSINGS = 1
if "MAX_DATABASE_CROSSINGS" not in globals():
    MAX_DATABASE_CROSSINGS = 13

results_all = []
timestamp = time.strftime("%Y%m%d-%H%M%S")

for tnum, target in enumerate(targets, start=1):
    idx = target["row_index"]
    knot_name = target["knot"]
    current_lower = target["lower"]
    current_upper = target["upper"]

    print("\n" + "="*80)
    print(f"[{tnum}/{len(targets)}] Processing:", knot_name, " current range:", [current_lower, current_upper])

    original_pd = parse_pd_cell(df.at[idx, pd_col])
    if original_pd is None:
        print("  Skipping: bad or missing PD.")
        results_all.append({
            "row_index": idx,
            "knot": knot_name,
            "status": "bad_pd",
        })
        continue

    base_vec = parse_vector_cell(df.at[idx, jones_col])
    if ensure_minmax_coeffs(base_vec) is None and len(original_pd) <= 20:
        new_vec, err = jones_vector_from_pd(original_pd)
        if new_vec is not None:
            df.at[idx, jones_col] = str(new_vec)

    variants = generate_inflated_variants(
        original_pd,
        num_variants=NUM_VARIANTS_PER_KNOT,
        backtrack_steps_min=BACKTRACK_STEPS_MIN,
        backtrack_steps_max=BACKTRACK_STEPS_MAX,
        riii_steps_max=RIII_STEPS_MAX,
    )

    best_new_upper = current_upper
    best_evidence = None
    tried = 0
    matched = 0

    for vnum, variant_pd in enumerate(variants, start=1):
        flip_variants = generate_single_flip_variants(variant_pd)

        for flip_index, flipped_pd in flip_variants:
            tried += 1
            success, min_crossings_found, best_pd = run_unknotter_on_pd(
                flipped_pd,
                model,
                cfg,
                episodes=UNKNOTTER_EPISODES_PER_FLIP,
                return_best_pd=True,
            )

            if best_pd is None:
                continue

            reduced_crossings = len(best_pd)

            # Only trust Jones matching inside the database crossing range.
            if not (MIN_DATABASE_CROSSINGS <= reduced_crossings <= MAX_DATABASE_CROSSINGS):
                continue

            vec, err = jones_vector_from_pd(best_pd)
            if vec is None:
                continue

            matches = match_jones_vector_to_database(vec)
            if not matches:
                continue

            matched += 1
            matched_upper = best_upper_bound_from_matches(matches)
            if matched_upper is None:
                continue

            candidate_upper = matched_upper + 1

            if candidate_upper < best_new_upper:
                best_new_upper = candidate_upper
                best_evidence = {
                    "variant_index": vnum,
                    "flip_index": int(flip_index),
                    "matched_upper": int(matched_upper),
                    "candidate_upper": int(candidate_upper),
                    "rl_success": bool(success),
                    "min_crossings_found": int(min_crossings_found),
                    "best_pd_crossings": reduced_crossings,
                    "matched_knots": sorted(
                        {(str(m["knot"]), int(m["upper"]), bool(m["mirror"])) for m in matches},
                        key=lambda x: (x[0], x[1], x[2])
                    ),
                    "best_pd": best_pd,
                    "jones_vector": vec,
                }
                print(f"  Improved via variant {vnum}, flip {flip_index}: upper {current_upper} -> {best_new_upper}")

    improved = best_new_upper < current_upper
    if improved:
        df.at[idx, u_col] = format_unknotting(current_lower, best_new_upper)

        if SAVE_EVERY_KNOT:
            df.to_excel(XLSX_PATH, index=False)
            print("  Saved updated workbook to:", XLSX_PATH)
    else:
        print("  No improvement found.")

    result = {
        "row_index": idx,
        "knot": knot_name,
        "old_lower": current_lower,
        "old_upper": current_upper,
        "new_upper": best_new_upper,
        "improved": improved,
        "tried_flip_runs": tried,
        "matched_flip_runs": matched,
        "evidence": best_evidence,
    }
    results_all.append(result)

    jsonl_path = OUT_DIR / "upper_bound_pass_results.jsonl"
    with open(jsonl_path, "a") as f:
        f.write(json.dumps(result) + "\n")

print("\nDone.")


[1/900] Processing: 12a_127  current range: [2, 3]
  No improvement found.

[2/900] Processing: 12a_144  current range: [3, 4]
  No improvement found.

[3/900] Processing: 12a_145  current range: [3, 4]
  No improvement found.

[4/900] Processing: 12a_146  current range: [3, 4]
  No improvement found.

[5/900] Processing: 12a_147  current range: [2, 4]
  No improvement found.

[6/900] Processing: 12a_148  current range: [2, 4]
  No improvement found.

[7/900] Processing: 12a_150  current range: [2, 4]
  No improvement found.

[8/900] Processing: 12a_152  current range: [2, 3]
  No improvement found.

[9/900] Processing: 12a_153  current range: [2, 3]
  No improvement found.

[10/900] Processing: 12a_156  current range: [2, 4]
  No improvement found.

[11/900] Processing: 12a_160  current range: [2, 4]
  No improvement found.

[12/900] Processing: 12a_161  current range: [2, 3]
  No improvement found.

[13/900] Processing: 12a_167  current range: [2, 4]
  No improvement found.

[14/900

KeyboardInterrupt: 

12a_49